# Which earlier token should this one look at?

In "the ball rolled because it was round," understanding "it" involves looking back at other words. Attention mixes information from tokens using weights. This notebook uses just **three tokens**, each represented by two numbers. It does not download or train a language model.

Before running: when predicting the first token, should it be allowed to look at tokens two and three? **No.** A **causal mask** crosses out those future positions. Run the cell and find the zeros above the diagonal of the printed weights table. See [the transformer lesson](../../docs/05-transformers-llms.md) for the arithmetic.

In [ ]:
import numpy as np
tokens = np.array([[1., 0.], [0., 1.], [1., 1.]])
queries = keys = values = tokens
scores = queries @ keys.T / np.sqrt(keys.shape[-1])
causal_scores = np.where(np.tril(np.ones(scores.shape, dtype=bool)), scores, -np.inf)
stable_scores = causal_scores - np.max(causal_scores, axis=-1, keepdims=True)
weights = np.exp(stable_scores)
weights /= weights.sum(axis=-1, keepdims=True)
context = weights @ values
assert scores.shape == weights.shape == (3, 3)
assert context.shape == tokens.shape
assert np.allclose(weights.sum(axis=1), 1)
assert np.allclose(np.triu(weights, k=1), 0)
print('weights:\n', weights.round(3), '\ncontext:\n', context.round(3))

## Check
Remove the causal mask: position 0 can now attend to positions 1 and 2, leaking future data in next-token training. Softmax must run across the key dimension; applying it across queries changes the meaning of each row.